# AI Jobbmatchning

Notebook-version av programmet. Koden är indelad i logiska celler för att göra projektet enklare att läsa, testa och förklara.

# 1. Importer
Innehåller mindre funktioner som används
på flera olika ställen i programmet.

In [ ]:
# requests hämtar annonser från JobTech. json hanterar datafiler.
import requests
import json
from datetime import datetime

print("Importer klara: JobTech-anrop, JSON-filer och datum/tid.")

# 2. Hjälpmetoder
Innehåller basklasser och underklasser
som hanterar data och information

In [ ]:
# Gör text jämförbar genom att ta bort mellanslag runtom och använda små bokstäver.
def normalisera(text):
    return text.lower().strip()

# Gör en lista med kompetenser till en enda läsbar textrad.
def visa_kompetenser(kompetenser):
    return ", ".join(kompetenser) if kompetenser else "Inga"

# Exempel på vad funktionerna gör.
print("Normaliserad text:", normalisera("  LÄRARE  "))
print("Kompetenser:", visa_kompetenser(["python", "samarbete"]))

# 3. Basklass och profil
Innehåller basklasser och underklasser
som hanterar data och information

In [ ]:
# Basklassen samlar information som både profilen och jobbannonserna använder.
class MatchningsData:
    def __init__(self):
        self.plats = []
        self.kompetenser = []


# Profilen ärver plats och kompetenser och lägger till önskade jobb.
class Profil(MatchningsData):
    def __init__(self):
        super().__init__()
        self.jobb = []

    def input_jobb(self):
        self._input("Vad hade du velat jobba som? ", "jobb")

    def input_plats(self):
        self._input("Vart hade du velat jobba? ", "plats")

    def input_kompetenser(self):
        self._input("Vad har du för kompetenser? ", "kompetenser")

    def _input(self, fråga, listnamn):
        # Dela upp flera svar med kommatecken och spara dem i rätt lista.
        print("\n" + "=" * 40)
        try:
            värden = [normalisera(värde) 
                      for värde in input(fråga).split(",") 
                      if värde.strip()]
            getattr(self, listnamn).extend(värden)

        except (KeyboardInterrupt, EOFError):

            print("\nInmatningen avbröts.")

# Visar vilka listor som skapas när en ny profil börjar.
print("Ny profil:", Profil().__dict__)

## 4. Jobbannons och matchare
Innehåller klasser som utför beräkningar,
matchningar och hanterar programmets logik.

In [ ]:
# Varje jobbannons sparar uppgifter som matcharen behöver.
class Jobbannons(MatchningsData):
    def __init__(self, jobb, plats, kompetenser, länk):
        super().__init__()
        self.jobb = jobb
        self.plats = plats
        self.kompetenser = kompetenser
        self.länk = länk


# Matcharen ger poäng för plats och kompetenser när jobbtiteln passar.
class Matchare:
    def __init__(self, profil, arbetsmarknad):
        self.profil = profil
        self.arbetsmarknad = arbetsmarknad

    def match(self):
        matchningar = []
        for jobb in self.arbetsmarknad:
            jobb_matchar = False
            for önskat_jobb in self.profil.jobb:
                if önskat_jobb in jobb.jobb:
                    jobb_matchar = True
                    break
            if not jobb_matchar:
                continue

            # Rätt plats ger två poäng och varje matchande kompetens ger en.
            poäng = 0
            for plats in self.profil.plats:
                if plats in jobb.plats:
                    poäng += 2
                    break
            for kompetens in self.profil.kompetenser:
                if kompetens in jobb.kompetenser:
                    poäng += 1
            if poäng == 0:
                poäng = 1
            matchningar.append((jobb, poäng))

        # Högst poäng ska visas först.
        matchningar.sort(key=lambda matchning: matchning[1], reverse=True)
        return matchningar

# Provar att en titel-, plats- och kompetensmatch ger tre poäng.
testprofil = Profil()
testprofil.jobb = ["lärare"]
testprofil.plats = ["stockholm"]
testprofil.kompetenser = ["planering"]
testannons = Jobbannons("lärare", "stockholm", ["planering"], "https://exempel.se")
testresultat = Matchare(testprofil, [testannons]).match()
print("Testresultat:", testresultat[0][0].jobb, "fick", testresultat[0][1], "poäng.")

## 5. Hämta jobb från JobTech API

Hämtar jobbannonser från JobTech Dev API,
skapar objekt och sparar undan rådata till fil

In [ ]:
# Hämtar annonser från JobTech utifrån jobb i profilen.
def hämta_jobb_från_api(profil):
    if not profil.jobb:
        print("Du måste ställa in ett önskat jobb först.")
        return []

    url = "https://jobsearch.api.jobtechdev.se/search"
    jobbannonser = []
    alla_data = []

    try:
        for sökord in profil.jobb:
            print("\nSöker efter:", sökord)
            response = requests.get(url, params={"q": sökord, "limit": 10}, timeout=10)
            print("Statuskod:", response.status_code)

            if response.status_code != 200:
                print("Kunde inte hämta annonser för detta sökord.")
                continue

            data = response.json()
            alla_data.append(data)
            träffar = data.get("hits", [])
            print("Antal träffar:", len(träffar))

            for jobb in träffar:
                titel = normalisera(jobb.get("headline") or "")
                adress = jobb.get("workplace_address") or {}
                plats = normalisera(adress.get("municipality") or "")
                länk = jobb.get("webpage_url") or ""
                beskrivning = jobb.get("description") or {}
                beskrivning = normalisera(beskrivning.get("text") or "")
                matchade_kompetenser = []

                for kompetens in profil.kompetenser:
                    if kompetens in beskrivning:
                        matchade_kompetenser.append(kompetens)

                jobbannonser.append(
                    Jobbannons(titel, plats, matchade_kompetenser, länk)
                )

        # Sparar API-svaret som rådata för att kunna granskas senare.
        try:
            with open("jobbdata.json", "w", encoding="utf-8") as fil:
                json.dump(alla_data, fil, ensure_ascii=False, indent=4)
        except OSError as fel:
            print("Kunde inte spara jobbdata:", fel)

        return jobbannonser

    except requests.exceptions.RequestException as fel:
        print("Ett fel uppstod när API:t kontaktades:", fel)
        return []
    except json.JSONDecodeError:
        print("Kunde inte läsa svaret från API:t.")
        return []

# Kontrollerar att sökningen stoppar säkert när profilen saknar jobb.
print("Resultat utan valt jobb:", hämta_jobb_från_api(Profil()))

## 6. Spara och visa historik

Hanterar sparande och visning av
tidigare sökningar och matchningar.

In [ ]:
# Sparar en sökning tillsammans med profilen och matchningarna.
def spara_historik(profil, matchningar):
    try:
        with open("historik.json", "r", encoding="utf-8") as fil:
            historik = json.load(fil)
    except (FileNotFoundError, json.JSONDecodeError):
        historik = []

    ny_sökning = {
        "datum": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "profil": {
            "jobb": profil.jobb,
            "plats": profil.plats,
            "kompetenser": profil.kompetenser
        },
        "matchningar": []
    }

    for jobb, poäng in matchningar:
        ny_sökning["matchningar"].append({
            "jobb": jobb.jobb,
            "plats": jobb.plats,
            "poäng": poäng,
            "kompetenser": jobb.kompetenser,
            "länk": jobb.länk
        })

    historik.append(ny_sökning)

    try:
        with open("historik.json", "w", encoding="utf-8") as fil:
            json.dump(historik, fil, ensure_ascii=False, indent=4)
    except OSError as fel:
        print("Kunde inte spara historiken:", fel)
        return

    print("Sökningen har sparats i historiken.")


# Läser historikfilen och skriver ut tidigare sökningar.
def visa_historik():
    try:
        with open("historik.json", "r", encoding="utf-8") as fil:
            historik = json.load(fil)
    except FileNotFoundError:
        print("Det finns ingen historik ännu.")
        return
    except json.JSONDecodeError:
        print("Historikfilen kunde inte läsas.")
        return

    if not historik:
        print("Det finns ingen historik ännu.")
        return

    print("\n" + "=" * 40 + "\nHISTORIK\n" + "=" * 40)
    for nummer, sökning in enumerate(historik, 1):
        print("\nSökning", nummer)
        print("Datum:", sökning["datum"])
        print("Jobb:", ", ".join(sökning["profil"]["jobb"]))
        print("Plats:", ", ".join(sökning["profil"]["plats"]))
        print("Kompetenser:", ", ".join(sökning["profil"]["kompetenser"]))
        print("Matchningar:")

        for jobb in sökning["matchningar"]:
            print("-", jobb["jobb"])
            print("  Plats:", jobb["plats"])
            print("  Poäng:", jobb["poäng"])
            print("  Matchande kompetenser:", visa_kompetenser(jobb["kompetenser"]))
            print("  Länk:", jobb["länk"])

        print("-" * 40)

# Bekräftar att funktionerna för att spara och visa historik finns.
print("Historikfunktioner:", callable(spara_historik), "och", callable(visa_historik))

## 7. Visa matchningar

Hämtar jobb från API, matchar dem mot profil,
sparar resultatet i historiken och skriver ut allt.

In [66]:
# Hämtar annonser, matchar dem mot profilen och visar resultaten.
def visa_matchningar(profil):
    print("Hämtar aktuella jobbannonser...")
    jobb_från_api = hämta_jobb_från_api(profil)
    if not jobb_från_api:
        print("Inga jobbannonser kunde hämtas.")
        return

    matchningar = Matchare(profil, jobb_från_api).match()
    if not matchningar:
        print("Inga jobb matchade din profil.")
        return

    spara_historik(profil, matchningar)
    print("\n" + "=" * 40 + "\nDINA MATCHNINGAR\n" + "=" * 40)
    for jobb, poäng in matchningar:
        print("\n-", jobb.jobb)
        print("  Plats:", jobb.plats)
        print("  Poäng:", poäng)
        print("  Matchande kompetenser:", visa_kompetenser(jobb.kompetenser))
        print("  Länk:", jobb.länk)

# Provar programmets skydd när profilen ännu saknar önskat jobb.
visa_matchningar(Profil())

Hämtar aktuella jobbannonser...
Du måste ställa in ett önskat jobb först.
Inga jobbannonser kunde hämtas.


## 8. Användargränssnitt

Skapar en profil-instans och hanterar
programmets huvudmeny i terminalen.

In [65]:
# Menyvalen använder den här profilen under programmets körning.
profil = Profil()

# Visar menyval och anropar funktionen som hör till användarens val.
def user_ui():
    while True:
        print("\n" + "=" * 40)
        print("AI-JOBBANALYS")
        print("=" * 40)
        print("1. Ställ in profil")
        print("2. Visa matchningar")
        print("3. Visa historik")
        print("4. Avsluta")

        try:
            svar = input("Välj ett alternativ 1-4: ")
        except (KeyboardInterrupt, EOFError):
            print("Programmet avslutas.")
            break

        if svar == "1":
            profil.input_jobb()
            profil.input_plats()
            profil.input_kompetenser()
            print("Profilen har uppdaterats.")
        elif svar == "2":
            visa_matchningar(profil)
        elif svar == "3":
            visa_historik()
        elif svar == "4":
            print("Tack för denna gång!")
            break
        else:
            print("Felaktigt val. Välj en siffra mellan 1 och 4.")

# Visar startprofilen och kontrollerar att menyn har skapats.
print("Startprofil:", profil.__dict__)
print("Menyfunktionen finns:", callable(user_ui))

Startprofil: {'plats': [], 'kompetenser': [], 'jobb': []}
Menyfunktionen finns: True


## 9. Starta programmet

Kör `user_ui()` i kodcellen nedan när du vill öppna den interaktiva menyn.
En vanlig körning av hela notebooken startar inte menyn automatiskt.

In [ ]:
# Startar programmets interaktiva meny. Välj 4 när du vill avsluta.
user_ui()
# Den här utskriften visas när menyn har avslutats.
print("Programmet har avslutats.")

## 10. Utvecklingsplan

1. Profil: Jobb, Plats, Kompetenser  
2. API: Hämta aktuella jobbannonser  
3. Matchning: Matcha jobb, plats, kompetenser  
4. Resultat: Visa jobb och matchningspoäng  
5. Förbättringar: Validering, Historik, Bättre UI  
6. Testning: Testa och färdigställ programmet